In [3]:
import polars as pl
import requests
import json
import numpy as np
import time
from tqdm import tqdm
import datetime

from constant import data_path
base_path = f"{data_path}/biorxiv"


In [3]:
import logging
import re
from dateutil.parser import parse as parse_date

logging.basicConfig(
    filename='log/fetch_log.log',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)
logging.info("Starting the script")

def clean_whitespace(d):
    def clean_string(s):
        # Replace any whitespace character (except normal space) with a normal space
        s = ''.join(c if ord(c) == 32 or not c.isspace() else ' ' for c in s)
        # Collapse multiple spaces and strip leading/trailing
        return re.sub(r'\s{2,}', ' ', s).strip()
    
    return {k: clean_string(v) if isinstance(v, str) else v for k, v in d.items()}
    
def normalize(article):
    article = clean_whitespace(article)
    if "authors" in article.keys(): article["authors"] = [author for author in article["authors"].split(";") if author.strip() != ""]
    if "preprint_authors" in article.keys(): article["preprint_authors"] = [author for author in article["preprint_authors"].split(";") if author.strip() != ""]
    if "category" in article.keys(): article["category"] = article["category"].strip()
    if "version" in article.keys(): article["version"] = int(article["version"])
    if "published" in article.keys(): article["published"] = article["published"] if article["published"]!="NA" else None
    return article

def fetch_articles(url):
    res = requests.get(url + "0")
    r = json.loads(res.text)

    if len(r["collection"])==0:
        logging.info(r["messages"])
        return
    
    total = int(r["messages"][0]["total"])
    if total == 0:
        logging.info("No articles found")
        return
    logging.info(f"Total articles found: {total}")

    for cursor in tqdm(range(int(np.ceil(total/100)))):
        try:
            res = requests.get(url + f"{cursor * 100}")
            logging.info(f"Fetching batch {cursor}")
            r = res.json()
            if len(r["collection"]) == 0:
                logging.info("No more articles found")
                break
            articles = [json.dumps(normalize(a)) for a in r["collection"]]
            yield articles
            time.sleep(1)
        except Exception as e:
            logging.error(f"Error fetching batch {cursor}: {e}")
    return 

def monthrange(any_day):
    next_month = any_day.replace(day=28) + datetime.timedelta(days=4)
    last_day_of_month = next_month - datetime.timedelta(days=next_month.day)
    first_day_of_month = any_day.replace(day=1)

    return first_day_of_month.strftime('%Y-%m-%d'), last_day_of_month.strftime('%Y-%m-%d')

In [ ]:

BATCH_SIZE = 1000

import os
for year in range(2013, 2025):
    start, end = f"{year}-01-01", f"{year}-12-31"
    buffer = []
    url = f"https://api.biorxiv.org/details/biorxiv/{start}/{end}/"
    # url = f"https://api.biorxiv.org/pubs/biorxiv/{start}/{end}/"
    
    path = f"{base_path}/full_{year}.jsonl"
    # Check if the file already exists
    if os.path.exists(path):
        logging.info(f"File {path} already exists, skipping...")
        continue
    else:
        logging.info(f"Creating file {path}...")
        os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "a", encoding="utf-8") as f:
        for article in fetch_articles(url):
            buffer.extend(article)
            if len(buffer) >= BATCH_SIZE:
                f.write("\n".join(buffer) + "\n")
                buffer.clear()

        if buffer:
            f.write("\n".join(buffer) + "\n")

In [7]:
BATCH_SIZE = 1000

import os

for year in range(2012, 2025):
    start, end = f"{year}-01-01", f"{year}-12-31"
    buffer = []
    url = f"https://api.biorxiv.org/pubs/biorxiv/{start}/{end}/"
    
    path = f"{base_path}/curated_detail_{year}.jsonl"
    # Check if the file already exists
    if os.path.exists(path):
        logging.info(f"File {path} already exists, skipping...")
        continue

    with open(path, "a", encoding="utf-8") as f:
        for article in fetch_articles(url):
            buffer.extend(article)
            if len(buffer) >= BATCH_SIZE:
                f.write("\n".join(buffer) + "\n")
                buffer.clear()

        if buffer:
            f.write("\n".join(buffer) + "\n")

100%|██████████████████████████████████████████████████████████| 187/187 [06:38<00:00,  2.13s/it]


# export all dois and curated dois

In [4]:
# concat all jsonl files vertically
df = pl.concat([
    pl.read_ndjson(f"{base_path}/full_{year}.jsonl")
    for year in range(2013, 2025)
]) 
df.head()

title,authors,author_corresponding,author_corresponding_institution,doi,date,version,type,license,category,jatsxml,abstract,funder,published,server
str,list[str],str,str,str,str,i64,str,str,str,str,str,str,str,str
"""Speciation and introgression b…","[""Yaniv Brandvain"", ""Amanda M Kenney"", … ""Andrea L Sweigart""]","""Yaniv Brandvain""","""Department of Evolution and Ec…","""10.1101/000109""","""2013-11-07""",1,"""New Results""","""cc_by""","""Evolutionary Biology""","""https://www.biorxiv.org/conten…","""Mimulus guttatus and M. nasutu…","""NA""","""10.1371/journal.pgen.1004410""","""bioRxiv"""
"""A Scalable Formulation for Eng…","[""Vanessa Jonsson"", ""Anders Rantzer"", ""Richard M Murray""]","""Vanessa Jonsson""","""Caltech""","""10.1101/000075""","""2013-11-07""",1,"""New Results""","""cc_by_nc""","""Evolutionary Biology""","""https://www.biorxiv.org/conten…","""It has been shown that optimal…","""NA""","""10.1109/ACC.2014.6859452""","""bioRxiv"""
"""Genome-wide targets of selecti…","[""Paolo Innocenti"", ""Ilona Flis"", ""Edward H Morrow""]","""Edward H Morrow""","""University of Sussex""","""10.1101/000240""","""2013-11-12""",1,"""New Results""","""cc_by""","""Evolutionary Biology""","""https://www.biorxiv.org/conten…","""Despite the common assumption …","""NA""",null,"""bioRxiv"""
"""Population genomics of paralle…","[""Nicola Nadeau"", ""Mayte Ruiz"", … ""Riccardo Papa""]","""Chri Jiggins""","""Cambridge""","""10.1101/000208""","""2013-11-12""",1,"""New Results""","""cc_by_nc_nd""","""Evolutionary Biology""","""https://www.biorxiv.org/conten…","""Hybrid zones can be valuable t…","""NA""","""10.1101/gr.169292.113""","""bioRxiv"""
"""The Origin of Human-infecting …","[""Liangsheng Zhang"", ""Zhenguo Zhang""]","""Zhenguo Zhang""","""Department of Biology, The Pen…","""10.1101/000398""","""2013-11-14""",1,"""New Results""","""cc_by_nc_nd""","""Evolutionary Biology""","""https://www.biorxiv.org/conten…","""In this study, we retraced the…","""NA""",null,"""bioRxiv"""


In [5]:
df_unique = df.unique("doi")

duplicate_published_dois = df_unique.filter(
    pl.col("published").is_not_null(),
    pl.col("published")!="",
    pl.col("published").is_duplicated()
).select("doi").to_series().unique().to_list()

print(df.height, df_unique.height)
print(len(duplicate_published_dois))
df.filter(
    pl.col("doi").is_null()
)

362441 262899
184


title,authors,author_corresponding,author_corresponding_institution,doi,date,version,type,license,category,jatsxml,abstract,funder,published,server
str,list[str],str,str,str,str,i64,str,str,str,str,str,str,str,str


In [6]:
df_cleaned = df_unique.filter(
    ~pl.col("doi").is_in(duplicate_published_dois)
)
df_cleaned

title,authors,author_corresponding,author_corresponding_institution,doi,date,version,type,license,category,jatsxml,abstract,funder,published,server
str,list[str],str,str,str,str,i64,str,str,str,str,str,str,str,str
"""Surviving hypothermia by ferri…","[""Pekec, T."", "" Lewandowski, J."", … "" Ciosk, R.""]","""Rafal Ciosk""","""University of Oslo and Institu…","""10.1101/2021.01.30.428937""","""2021-02-01""",1,"""new results""","""cc_no""","""physiology""","""https://www.biorxiv.org/conten…","""How animals rewire cellular pr…","""NA""","""10.1038/s41467-022-32500-z""","""bioRxiv"""
"""Human cytomegalovirus UL7, miR…","[""Hancock, M. H."", "" Crawford, L. B."", … "" Caposio, P.""]","""Patrizia Caposio""","""Oregon Health & Science Univer…","""10.1101/2020.09.28.317859""","""2020-09-29""",1,"""new results""","""cc_no""","""microbiology""","""https://www.biorxiv.org/conten…","""Human cytomegalovirus (HCMV) i…","""NA""","""10.1128/msphere.00986-20""","""bioRxiv"""
"""SARS-CoV-2 Viral Genes Comprom…","[""Liu, J."", "" Zhang, Y."", … "" Yang, L.""]","""Lei Yang""","""Indiana University School of M…","""10.1101/2022.01.20.477147""","""2022-01-23""",1,"""new results""","""cc_by_nc_nd""","""molecular biology""","""https://www.biorxiv.org/conten…","""Cardiac manifestations are com…","""NA""",null,"""bioRxiv"""
"""Follicle stimulating hormone s…","[""Torzone, S. K."", "" Park, A. Y."", … "" Dowen, R. H.""]","""Robert H Dowen""","""University of North Carolina a…","""10.1101/2023.01.07.523122""","""2023-01-08""",1,"""new results""","""cc_no""","""developmental biology""","""https://www.biorxiv.org/conten…","""Animals integrate developmenta…","""NA""",null,"""bioRxiv"""
"""Direct thin-layer agar for bed…","[""Cuella Martin, I."", "" Runyambo, D."", … "" Rigouts, L.""]","""Isabel Cuella Martin""","""Instituut voor Tropische Genee…","""10.1101/2024.11.18.624108""","""2024-11-20""",1,"""new results""","""cc_by_nc_nd""","""microbiology""","""https://www.biorxiv.org/conten…","""This study evaluated thin-laye…","""NA""","""10.1128/jcm.01806-24""","""bioRxiv"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Comprehensive evaluation of ha…","[""Wang, Y.-w."", "" Chen, X."", "" Yan, C.-G.""]","""Chao-Gan Yan""","""Institute of Psychology, Chine…","""10.1101/2022.09.22.508637""","""2022-09-23""",1,"""new results""","""cc_no""","""neuroscience""","""https://www.biorxiv.org/conten…","""To embrace big-data neuroimagi…","""NA""","""10.1016/j.neuroimage.2023.1200…","""bioRxiv"""
"""The genomic landscape of polym…","[""Gargi Dayama"", ""Sarah B Emery"", … ""Ryan E Mills""]","""Ryan E Mills""","""University ofMichigan""","""10.1101/008144""","""2014-08-19""",1,"""New Results""","""cc_by_nc_nd""","""Genomics""","""https://www.biorxiv.org/conten…","""The transfer of mitochondrial …","""NA""","""10.1093/nar/gku1038""","""bioRxiv"""
"""A cross-species analysis of ne…","[""Pham, L."", "" Guma, E."", … "" Raznahan, A.""]","""Armin Raznahan""","""NIMH""","""10.1101/2024.11.05.622111""","""2024-11-05""",1,"""new results""","""cc_by""","""neuroscience""","""https://www.biorxiv.org/conten…","""Structural covariance in brain…","""NA""","""10.1186/s13293-025-00728-1""","""bioRxiv"""


In [8]:
(
    df_cleaned
    # .group_by("id").agg(
    #     pl.col("version").sort_by("version")
    # )
    .with_columns(
        pl.col("published").alias("curated_doi"),
        pl.col("category").alias("primary_category"),
        pl.col("date").alias("published_date"),
    )
    .select([
        # pl.col("id"),
        pl.col("doi"),
        # pl.col("categories"),
        pl.col("primary_category"),
        pl.col("curated_doi"),
        pl.col("published_date"),
        # pl.col("last_updated_date"),
    ]).write_ndjson(f"{data_path}/biorxiv/biorxiv_preprints.dedup.jsonl")
)

In [60]:
import json

curated_doi = df_cleaned.filter(
    pl.col("published").is_not_null(),
    pl.col("published")!=""
).select("doi", "published")
curated_doi_dict = dict(zip(curated_doi["doi"], curated_doi["published"]))

unique_doi = df_cleaned.select("doi").to_series().unique().to_list()

print(len(curated_doi_dict), len(unique_doi))

with open(f"{base_path}/curated_biorxiv.json", "w") as f:
    json.dump(curated_doi_dict, f)
    
with open(f"{base_path}/biorxiv_unique_doi.json", "w") as f:
    json.dump(unique_doi, f)

149892 262715
